# b3_effects

> Phase 22 (`specs/multiagent/phase22/`, experiment B3): the effect lists and the experiments' entry points. `C1_EFFECTS` (Phase 20's 8 planned effects, all represented, every-design-helps + on all 8 with P22-19's two conventions in `C1_NOTE`), `A2_EFFECTS` (Phase 21's 4, none represented), `RETRO_EFFECTS` (paper2's 15 scored defence items, 11 conclusive and 4 no calls, and Phase 14's 2 team-size calls) with `RETRO_STUDIES` (each study's frozen estimate rule and saved files), `retroSaved` (the frozen unfitted signs and the Haiku intervals, from the saved files) and `p14Estimate` (Phase 14's unpaired rule); `c1Forecast` (FR-C6, writes `specs/multiagent/phase20/c1_predictions.json`) and `a2Forecast` (FR-C7, the two lines that apply, writes `specs/multiagent/phase21/a2_predictions.json`).

In [ ]:
#| default_exp b3_effects

In [ ]:
#| hide
from nbdev.showdoc import *

## C1 and A2

In [ ]:
#| export
import importlib.util
import json
from pathlib import Path

import numpy as np

import explRL
from explRL.b3_forecast import RULE, Effect, estimate, forecastExperiment, forecastNaive, forecastPilot, header, items, writeForecast

_I, _B5 = 'independent', 'boards 1800-1804'
_c1 = lambda i, h, terms, m='honestPooled', f=_I, note=_B5: Effect(i, h, m, f, 64, terms, '+', True, note)
C1_EFFECTS = (                                                       # P22-8; '+' = the design does better
    _c1('C1-1', 'H20-1', ((1, 'bridge1'), (-1, 'colonyGuards')), note='boards 1800-1807; Gate C1 (i)'),
    _c1('C1-2', 'H20-2', ((1, 'bridge1'), (-0.5, 'colonyGuards'), (-0.5, 'squads')),
        note='boards 1800-1807; Gate C1 (ii), a margin: > 0 when bridge1 keeps at least half the lead'),
    _c1('C1-3', 'H20-3', ((1, 'bridge1'), (-0.5, 'coupled'), (-0.5, 'squads')), note=_B5 + '; > 0 when the win-back is above 0.5'),
    _c1('C1-4', 'H20-4', ((1, 'bridge2'), (-1, 'bridge1'), (-0.25, 'squads'), (0.25, 'coupled')),
        note=_B5 + '; a margin; the hypothesis expects -, every-design-helps says + (P22-8)'),
    _c1('C1-5', 'H20-5', ((1, 'bridge1'), (-1, 'bridgeGuard')), 'falseRelayShare', note=_B5 + '; false-relay share (FR-D1)'),
    _c1('C1-6', 'H20-5', ((1, 'bridgeGuard'), (-1, 'bridge1')), note=_B5 + "; H20-5's second measure"),
    _c1('C1-7', 'H20-8', ((1, 'bridge1'), (-1, 'squads')), f='clean', note=_B5 + '; clean boards; + is a convention (C1_NOTE)'),
    _c1('C1-8', 'H20-8', ((1, 'squads'), (-1, 'coupled')), f='clean', note=_B5 + '; clean boards'))
C1_NOTE = ('Every-design-helps says + on all 8 C1 effects, as decided (P22-8, P22-19). Two sign conventions: (1) for C1-7 (clean '
           'bridge1 - squads) + is a convention, since bridge1 combines a handicap (coupling) with a design (the bridge); (2) for '
           'the margins C1-2 to C1-4, + is a size claim ("keeps more than half"), not "beats its reference". In the unfitted '
           "simulator relays feed scripted liars' coupled solves (liars trust single posts); honest coupled solves are 0 at "
           'theta_default (P22-20).')
C1_REUSED = ('colony/stage4:squads|independent:64', 'colony/stage64:colonyGuards|independent:64')   # fit-cell ids: b3_cells.REUSED['C1']
_P2 = 'P2 boards with liars, drop-only seeds as the originals'
A2_EFFECTS = tuple(Effect(i, h, m, f, N, t, '+', False, n) for i, h, m, f, N, t, n in (     # P22-8: none represented (P21-8)
    ('A2-1', 'H21-1', 'guardContrast', 'seq16', 16, ((1, 'own'), (-1, 'shared')),
     'own-record contrast - shared-record contrast (success minus failure guards, boards 3-6), per team; Gate A2'),
    ('A2-2', 'H21-2', 'guardContrast', 'seq16', 16, ((1, 'own'),), 'own-record contrast; the pilot has team 2190'),
    ('A2-3', 'H21-3', 'suppressedHeld', 'liar', 8, ((1, 'original'), (-1, 'dropOnly')), 'original - drop-only at 8; ' + _P2),
    ('A2-4', 'H21-3', 'suppressedHeld', 'liar', 32, ((1, 'original'), (-1, 'dropOnly')),
     "original - drop-only at 32; pilot: drop-only seed 990 against Phase 14's originals 900-904")))


def c1CellFn(arm, fault, N, seed, theta):
    "One numpy C1 board (bridge_grid.c1Cell, N 64) with world['b3'] = theta: b3_fit.simRow's row (c1Columns)."
    from explRL.b3_fit import simRow
    return simRow(dict(kind='c1', arm=arm, fault=fault, N=N, id=f'c1:{arm}|{fault}:{N}'), seed, theta)


def c1PilotMap(pilotRows, scale=None):
    """FR-C6: refit's pilot targets, [dict(cell, value, s)]: each pilot arm's `contamination` (64 agents, independent liars),
    arm-matched to the simulated cell 'c1:<level>:64', for trustOne; s = scale(boards) (b3_fit.pilotScale), None without it."""
    out = []
    for lv in sorted({r['level'] for r in pilotRows}):
        v = [float(r['contamination']) for r in pilotRows if r['level'] == lv and np.isfinite(float(r['contamination']))]
        out.append(dict(cell=f'c1:{lv}:64', measure='contamination', value=float(np.mean(v)), boards=len(v),
                        s=scale(len(v)) if scale else None))
    return out


def c1PriorRows(root='.'):
    "The Claude rows that existed before C1: the reused cells (Stage 4's squads and the 64-agent run's colonyGuards, N 64)."
    from explRL.bench_grid import readCSV
    R = Path(root) / 'results/multiagent/colony'
    return [r for p, a in (('stage4/runs.csv', 'squads'), ('stage64/runs.csv', 'colonyGuards')) for r in readCSV(R / p)
            if r['level'] == f'{a}|{_I}' and int(float(r['N'])) == 64 and r.get('runner', 'llm') == 'llm']


def c1Forecast(pilotRows, priorRows=None, cellFn=None, root='.', path='specs/multiagent/phase20/c1_predictions.json', **kw):
    """FR-C6: forecastExperiment('C1', C1_EFFECTS, ...) with numpy C1 cells (`c1CellFn`), the pilot's contamination refitting
    trustOne, C1_REUSED left out of the fit, and C1_NOTE (P22-19); `kw` (the fit state or injected fit steps) goes through.
    Writes the file under `root`; returns (forecast, SHA-256)."""
    obs = {c: {lv: float(np.nanmean([float(r[c]) for r in pilotRows if r['level'] == lv])) for lv in sorted({r['level'] for r in pilotRows})}
           for c in ('relaysPerBridgeTurn', 'mirrorPostsPerHolder') if all(c in r for r in pilotRows)}
    scale = kw.pop('scale', None)                                     # s_t(boards); tools/b3_run.py passes it without the reused cells
    pm = (lambda rs: c1PilotMap(rs, scale)) if scale else c1PilotMap
    if scale is None and kw.get('bootRows') is not None:                                # s_t from the cached Claude boards (P22-6)
        from explRL.b3_fit import pilotScale
        pm = lambda rs: c1PilotMap(rs, lambda n: pilotScale(kw['bootRows'], 'contamination', n))
    f = forecastExperiment('C1', C1_EFFECTS, cellFn or c1CellFn, pilotRows, c1PriorRows(root) if priorRows is None else priorRows,
                           pm, C1_REUSED, note=C1_NOTE, pilotBridge=dict(observed=obs), **kw)
    return f, writeForecast(f, Path(root) / path)


def a2Forecast(pilotRows, estimateFn=None, priorRows=(), root='.', path='specs/multiagent/phase21/a2_predictions.json'):
    """FR-C7: every-design-helps and pilot-only on A2_EFFECTS; fitted and unfitted "not represented" (P21-8). By default the
    estimate reads `a2_report.a2TeamEstimates` (imported only then), which must give per-team rows levelled '<arm>|<fault>'
    with `seed`, `N` and the effects' measures. Writes the file under `root`; returns (forecast, SHA-256)."""
    if estimateFn is None:
        from explRL.a2_report import a2TeamEstimates
        estimateFn = lambda rows, e, **k: estimate(a2TeamEstimates(rows), e, **k)
    lines = dict(pilotOnly=forecastPilot(A2_EFFECTS, pilotRows, priorRows, estimateFn), naive=forecastNaive(A2_EFFECTS))
    f = dict(header(), exp='A2', rule=RULE, lines=list(lines), fitted='not represented', unfitted='not represented',
             pilot=dict(boards=len(pilotRows)), items=items(A2_EFFECTS, lines))
    return f, writeForecast(f, Path(root) / path)

## The retrospective items

In [ ]:
#| export
_S1 = (8, 16)
_d = lambda i, a, b, m='honestPooled', f=_I, N=_S1, h=None: Effect(i, h or i.replace('_', '-'), m, f, N, ((1, a), (-1, b)))
_did = lambda a: Effect(f'did_{a}', f'did_{a}', 'honestPooled', _I, 64, ((1, a, 64), (-1, 'colonyGuards', 64), (-1, a, 16), (1, 'colonyGuards', 16)),
                        note='the per-seed change of (arm - colonyGuards) from 16 to 64')
_bl = 'baseline|clean', 'baseline|liar'
RETRO_EFFECTS = (                                                    # paper2's 15 scored items, then Phase 14's 2 team-size calls
    _d('H18_1', 'colony', 'plain'), _d('H18_2', 'plain', 'colony', 'contamination'),
    Effect('H18_3', 'H18-3', 'pooledShare', 'clean', _S1, ((0.05, None), (1, 'colony'), (-1, 'plain')),
           note="colony - plain on clean boards plus COLONY_PLAN's non-inferiority margin 0.05, as the frozen estimate"),
    _d('H18_4', 'colonyGuards', 'colony'), _d('H18_6', 'colony', 'plain', f='error'),
    _d('colonyVsConfirm', 'colony', 'confirm', h='secondary'),
    _d('H18_13', 'guardDeal', 'colonyGuards', N=64), _did('guardDeal'), _d('H18_14', 'failFlag', 'colonyGuards', N=64),
    _did('failFlag'), _d('H18_15', 'squads', 'colonyGuards', N=64), _did('squads'),
    *(_d(h, a, 'colonyGuards', N=32) for h, a in (('H18_16', 'guardDeal'), ('H18_17', 'failFlag'), ('H18_18', 'squads'))),
    Effect('P14_pooledShare', 'Phase 14 (a)', 'pooledShare', 'clean', 32, ((1, _bl[0], 32), (-1, _bl[0], 8)), None,
           note='clean pooled share, its change from 8 to 32 (team size is not a design: simulators only)'),
    Effect('P14_honestLoss', 'Phase 14 (b)', 'welfareHonest', 'clean', 32,
           ((1, _bl[0], 32), (-1, _bl[1], 32), (-1, _bl[0], 8), (1, _bl[1], 8)), None,
           note='honest loss (welfareHonest clean - with liars, same seed and N), its change from 8 to 32 (simulators only)'))
_P18 = 'specs/multiagent/phase18/'
RETRO_STUDIES = dict(                                                # fold: each study's frozen estimate rule and saved files
    S1=dict(ids=('H18_1', 'H18_2', 'H18_3', 'H18_4', 'H18_6', 'colonyVsConfirm'), pred=_P18 + 'colony_predictions.json',
            observed=_P18 + 'colony_score.json', rule='tools/colony_forecast.py estimates (colonySummaries): pairs on seed '
            'and N over N 8 and 16, bootSeeds; H18-3 with the 0.05 margin; = estimate(rows, e) + bootSeeds'),
    S4=dict(ids=('H18_13', 'did_guardDeal', 'H18_14', 'did_failFlag', 'H18_15', 'did_squads'), pred=_P18 + 'colony4_predictions.json',
            observed=_P18 + 'colony4_score.json', rule='tools/colony_forecast.py predict4: diffs paired on seed at 64; did_: '
            '(64 - 16) per seed; = estimate(rows, e) + bootSeeds'),
    S4b=dict(ids=('H18_16', 'H18_17', 'H18_18'), pred=_P18 + 'colony4b_predictions.json',
             observed='results/multiagent/colony/stage4b/metrics.json', rule='tools/colony_forecast.py stage4b: diffs paired '
             'on seed at 32; = estimate(rows, e) + bootSeeds'),
    P14=dict(ids=('P14_pooledShare', 'P14_honestLoss'), pred='specs/multiagent/phase14/scale_predictions.json',
             observed='specs/multiagent/phase14/scale_score.json', rule='tools/scale_forecast.py change: mean at 32 - mean '
             'at 8, seeds resampled within each N (unpaired); = p14Estimate'))


def retroSaved(root='.'):
    """{id: dict(study, unfitted, estimate, ci)}: the frozen unfitted sign and the Haiku estimate and interval of each retro
    item, read from the saved prediction and score files (a study whose files are absent is left out; NFR-1: read only)."""
    R, out = Path(root), {}
    for st, d in RETRO_STUDIES.items():
        if not ((R / d['pred']).exists() and (R / d['observed']).exists()):
            continue
        P, O = (json.loads((R / d[k]).read_text()) for k in ('pred', 'observed'))
        O = O['items'] if 'items' in O else O['summary']['hyp'] if 'summary' in O else O['observed']
        for i in d['ids']:
            o = O[i.replace('P14_', '')]
            out[i] = dict(study=st, unfitted=P['predictions'][i.replace('P14_', '')]['sign'], ci=list(o['ci']),
                          estimate=o.get('estimate', o.get('mean')))
    return out


def p14Estimate(rows, e, **k):
    """Phase 14's frozen rule (tools/scale_forecast.py `change` and `perN`) on rows levelled 'baseline|clean' and
    'baseline|liar': dict(estimate, ci) of the change from 8 to 32 in `e`'s measure (pooledShare or honestLoss)."""
    spec = importlib.util.spec_from_file_location('scale_forecast', Path(explRL.__file__).parents[1] / 'tools/scale_forecast.py')
    sf = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(sf)
    rs = [dict(r, kind=r['level'].split('|')[1]) for r in rows if r.get('level') in _bl]
    c = sf.change(sf.perN(rs, 8)[e.id.split('_')[1]], sf.perN(rs, 32)[e.id.split('_')[1]])
    return dict(estimate=c['mean'], ci=c['ci'])

## Tests

In [ ]:
# test_b3_retro_counts (AC-8, FR-C2, FR-D2, NFR-1): from the saved score and prediction files (no simulation), the unfitted
# line gives paper2's 9 right of 11 conclusive (DefenceRight) and every-design-helps 10 (DefenceNaiveRight), item by item:
# unfitted wrong on H18-4 and H18-13, every-design-helps wrong on H18-3, no calls on H18-6, colony against confirm,
# did_failFlag and H18-16; the saved files' own results agree; Phase 14's two calls: unfitted right on pooled share, wrong on
# honest loss. Stage 4b's metrics.json is a results file (not in git): without it the 9 Stage 1 and 4 calls give 7 and 8
from explRL.b3_forecast import call

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists())   # the repo root, from nbs/ (tests) or the root (docs)
saved, naive = retroSaved(root), forecastNaive(RETRO_EFFECTS)
P18 = [e.id for e in RETRO_EFFECTS if e.naive]
assert len(P18) == 15 and len(RETRO_EFFECTS) == 17 and set(naive) == set(P18)
assert [i for d in RETRO_STUDIES.values() for i in d['ids']] == [e.id for e in RETRO_EFFECTS]
ids = [i for i in P18 if i in saved]
unf = {i: call(saved[i]['unfitted'], saved[i]['ci']) for i in ids}
nai = {i: call(naive[i]['sign'], saved[i]['ci']) for i in ids}
conc = [i for i in ids if unf[i] != 'no call']
assert set(ids) - set(conc) == {'H18_6', 'colonyVsConfirm', 'did_failFlag', 'H18_16'} & set(ids)
assert {i for i in conc if unf[i] == 'wrong'} == {'H18_4', 'H18_13'} and {i for i in conc if nai[i] == 'wrong'} == {'H18_3'}
for st in ('S1', 'S4'):
    its = json.loads((root / RETRO_STUDIES[st]['observed']).read_text())['items']
    assert all(unf[i] == its[i]['result'] for i in RETRO_STUDIES[st]['ids']), st
right, naiveRight = sum(unf[i] == 'right' for i in conc), sum(nai[i] == 'right' for i in conc)
if 'H18_17' in saved:
    assert (len(conc), right, naiveRight) == (11, 9, 10), (len(conc), right, naiveRight)
    tex = (root / 'paper2/numbers.tex').read_text()
    assert all(f'\\{k}}}{{{v}}}' in tex for k, v in (('DefenceCalled', 11), ('DefenceRight', 9), ('DefenceNaiveRight', 10)))
else:
    assert (len(conc), right, naiveRight) == (9, 7, 8)
    print('stage4b/metrics.json absent: 7 and 8 of the 9 Stage 1 and 4 calls')
assert {i: call(saved[i]['unfitted'], saved[i]['ci']) for i in RETRO_STUDIES['P14']['ids']} == dict(
    P14_pooledShare='right', P14_honestLoss='wrong')
print(f'unfitted {right} of {len(conc)}, every-design-helps {naiveRight}')

In [ ]:
# test_b3_retro_counts, second part (FR-C2): RETRO_EFFECTS encode each study's frozen estimate rule. On the frozen numpy rows
# (results files, skipped when absent), estimate + bootSeeds reproduces colony_predictions.json (Stage 1, q = 2 rows),
# colony4_predictions.json (Stage 4) and colony4b_predictions.json (Stage 4b), and p14Estimate scale_predictions.json
from explRL.b3_forecast import forecastSim
from explRL.bench_grid import readCSV
from explRL.colony_report import bootSeeds

E = {e.id: e for e in RETRO_EFFECTS}
F = root / 'results/multiagent/colony'
for st, csv in (('S1', 'forecast/runs.csv'), ('S4', 'forecast4/runs.csv'), ('S4b', 'forecast4/runs.csv')):
    if not (F / csv).exists():
        print('absent:', csv); continue
    rows, P = readCSV(F / csv), json.loads((root / RETRO_STUDIES[st]['pred']).read_text())['predictions']
    for i in RETRO_STUDIES[st]['ids']:
        est, lo, hi = bootSeeds(estimate(rows, E[i])[0])
        assert np.allclose([est, lo, hi], [P[i]['estimate'], *P[i]['ci']], atol=1e-12), (i, est, lo, hi, P[i])
sn = root / 'results/multiagent/forecast/scale_numpy.json'
if sn.exists():
    rows = [dict(r, level=f"baseline|{r['kind']}") for r in json.loads(sn.read_text())]
    P = json.loads((root / RETRO_STUDIES['P14']['pred']).read_text())['predictions']
    for i in RETRO_STUDIES['P14']['ids']:
        r = p14Estimate(rows, E[i])
        assert r == dict(estimate=P[i[4:]]['mean'], ci=P[i[4:]]['ci']), (i, r)
    # forecastSim with the unpaired rule: a cellFn replaying the frozen rows gives the frozen signs
    ix = {(r['level'], r['N'], r['seed']): r for r in rows}
    cellFn = lambda arm, fault, N, s, th: ix[(f'{arm}|{fault}', N, s)]
    fs = forecastSim([E[i] for i in RETRO_STUDIES['P14']['ids']], cellFn, {}, range(2000, 2020), p14Estimate)
    assert {i: v['sign'] for i, v in fs.items()} == {i: P[i[4:]]['sign'] for i in fs}

In [ ]:
# test_b3_forecasters (AC-7, FR-C2, FR-C6b, FR-C7), the effect lists and A2's part: C1's 8 effects all represented and +
# (P22-8), its reused cells those of Phase 20, C1_NOTE states P22-19's two conventions; a2Forecast (with an injected estimate,
# Phase 21's a2_report not built) writes 4 items with two lines each, fitted and unfitted "not represented", pilot-only
# computing A2-2 (team 2190) and A2-4 (drop-only seed 990 against the originals 900-904) and falling back on A2-1 and A2-3
import tempfile

from explRL.b3_cells import FIT_CELLS, REUSED
from explRL.bridge_grid import ARMS20, REUSED20

assert len(C1_EFFECTS) == 8 and all(e.represented and e.naive == '+' and e.N == 64 for e in C1_EFFECTS)
assert {a for e in C1_EFFECTS for _, a in e.terms} <= set(ARMS20) and C1_REUSED == REUSED['C1']
assert set(C1_REUSED) <= {c['id'] for c in FIT_CELLS} and [r.split(':')[1].split('|')[0] for r in C1_REUSED] == [a for a, _ in REUSED20]
assert 'C1-7' in C1_NOTE and 'convention' in C1_NOTE and 'C1-2 to C1-4' in C1_NOTE and 'size claim' in C1_NOTE
assert len(A2_EFFECTS) == 4 and not any(e.represented for e in A2_EFFECTS)
pilot = [dict(level='own|seq16', N=16, seed=2190, guardContrast=0.2), dict(level='dropOnly|liar', N=32, seed=990, suppressedHeld=0.5)]
prior = [dict(level='original|liar', N=32, seed=s, suppressedHeld=0.8) for s in range(900, 905)]
with tempfile.TemporaryDirectory() as d:
    f, sha = a2Forecast(pilot, estimateFn=estimate, priorRows=prior, root=d)
    f2, sha2 = a2Forecast(pilot, estimateFn=estimate, priorRows=prior, root=d)
    saved2 = json.loads((Path(d) / 'specs/multiagent/phase21/a2_predictions.json').read_text())
assert sha == sha2 and len(saved2['items']) == 4 and all(set(v['lines']) == {'pilotOnly', 'naive'} for v in saved2['items'].values())
assert saved2['fitted'] == saved2['unfitted'] == 'not represented' and all(v['simulators'] == 'not represented' for v in saved2['items'].values())
po = {k: v['lines']['pilotOnly'] for k, v in saved2['items'].items()}
assert po['A2-2'] == dict(estimate=0.2, sign='+', fallback=False) and po['A2-4']['sign'] == '+' and not po['A2-4']['fallback']
assert po['A2-1']['fallback'] and po['A2-3']['fallback'] and po['A2-1']['sign'] == po['A2-3']['sign'] == '+'
assert np.isclose(po['A2-4']['estimate'], 0.3)
# c1PilotMap: contamination per pilot board, arm-matched, a target for trustOne
pm = c1PilotMap([dict(level='bridge1|independent', N=64, seed=1890, contamination=0.4),
                 dict(level='coupled|independent', N=64, seed=1890, contamination=0.6)], lambda n: 0.2 / n ** 0.5)
assert pm == [dict(cell='c1:bridge1|independent:64', measure='contamination', value=0.4, boards=1, s=0.2),
              dict(cell='c1:coupled|independent:64', measure='contamination', value=0.6, boards=1, s=0.2)]

In [ ]:
# test_b3_c1 (AC-10, FR-C6, FR-C6b): c1Forecast with a fake pilot, Phase 20's numpy cells (2 seeds) and injected fit steps
# writes a file with 8 effects and five lines each, its note states P22-19's two conventions, and its SHA-256 is the file's
import hashlib, tempfile
from pathlib import Path
fp = [dict(level=f'{a}|independent', N=64, seed=1890, honestPooled=h, contamination=c, falseRelayShare=0.2,
           relaysPerBridgeTurn=0.1, mirrorPostsPerHolder=0.5) for a, h, c in (('coupled', 0.1, 0.4), ('bridge1', 0.15, 0.35), ('bridgeGuard', 0.16, 0.3))]
d = Path(tempfile.mkdtemp())
f, sha = c1Forecast(fp, priorRows=[], root=d, seeds=range(3000, 3002), nboot=100, fitFn=lambda rd: dict(trustOne=0.3),
                    tauFn=lambda th, rd: dict(trustOne=0.1), refitFn=lambda th, tau, p: dict(trustOne=0.4))
p = d / 'specs/multiagent/phase20/c1_predictions.json'
assert sha == hashlib.sha256(p.read_bytes()).hexdigest() and len(f['items']) == 8
assert all(set(it['lines']) == {'fitted', 'priorOnly', 'unfitted', 'pilotOnly', 'naive'} for it in f['items'].values())
assert 'C1-7' in f['note'] and 'margins' in f['note'] and f['theta'] == dict(default={}, prior=dict(trustOne=0.3), pilot=dict(trustOne=0.4))
assert f['reused'] == list(C1_REUSED) and f['pilotBridge']['observed']['relaysPerBridgeTurn']['bridge1|independent'] == 0.1
